# Week 8 — IPW & Mahalanobis Matching (Standalone Notebook)
This notebook computes:
1. **Q1:** IPW ATE on `homework_8.1.csv`
2. **Q2:** First three propensity scores
3. **Q3:** Mahalanobis matching ATE on `homework_8.2.csv` (with replacement)
4. **Q4:** Least common support treated item (farthest nearest-neighbor Mahalanobis distance)

**Instructions**
- Put `homework_8.1.csv` and `homework_8.2.csv` in the **same folder** as this notebook.
- Run cells top-to-bottom.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.spatial.distance import mahalanobis

pd.set_option('display.precision', 4)
DATA1 = 'homework_8.1.csv'
DATA2 = 'homework_8.2.csv'

## Q1 & Q2 — IPW on `homework_8.1.csv`

In [ ]:
# Load data
df = pd.read_csv(DATA1)

# Logistic regression: X ~ Z  -> propensity score p = P(X=1|Z)
X = sm.add_constant(df['Z'])
y = df['X']
logit_model = sm.Logit(y, X).fit(disp=0)
df['p'] = logit_model.predict(X)

# Inverse probability weights
df['w'] = np.where(df['X'] == 1, 1 / df['p'], 1 / (1 - df['p']))

# Weighted group means and ATE
treated = df[df['X'] == 1]
control = df[df['X'] == 0]
Y1 = np.average(treated['Y'], weights=treated['w'])
Y0 = np.average(control['Y'], weights=control['w'])
ATE_ipw = Y1 - Y0

print('IPW ATE (Q1):', round(float(ATE_ipw), 3))
print('First three propensity scores (Q2):', df['p'].head(3).round(2).tolist())

## Q3 — Mahalanobis matching on `homework_8.2.csv` (with replacement)

In [ ]:
# Load data
df2 = pd.read_csv(DATA2)

treated2 = df2[df2['X'] == 1].copy()
control2 = df2[df2['X'] == 0].copy()

# Inverse covariance matrix for [Z1, Z2] from full sample
Z = df2[['Z1','Z2']].to_numpy()
inv_cov = np.linalg.inv(np.cov(Z.T))

def mahalanobis_vectorized(u, V_inv, X):
    diff = X - u
    left = diff @ V_inv
    return np.sqrt(np.sum(left * diff, axis=1))

treated_Z = treated2[['Z1','Z2']].to_numpy()
treated_Y = treated2['Y'].to_numpy()
control_Z = control2[['Z1','Z2']].to_numpy()
control_Y = control2['Y'].to_numpy()

# Nearest-neighbor matching (with replacement)
diffs = []
nearest_dists = []
for i, u in enumerate(treated_Z):
    dists = mahalanobis_vectorized(u, inv_cov, control_Z)
    j = np.argmin(dists)
    diffs.append(treated_Y[i] - control_Y[j])
    nearest_dists.append(dists[j])

ATE_mahal = float(np.mean(diffs))
print('Mahalanobis ATE (Q3):', round(ATE_mahal, 3))

## Q4 — Least common support treated item

In [ ]:
# Treated unit whose nearest control is farthest away (least common support)
idx_farthest = int(np.argmax(nearest_dists))
least_support_point = treated2.iloc[idx_farthest][['Z1','Z2']]
print('Least common support treated (Z1, Z2):', tuple(least_support_point.round(2)))